Fundamentals of Deep Learning Models

# Lab 02-2: Logistic Regression
## Exercise: Predicting Iris Species

This exercise implements **binary logistic regression** (Section 2.4) to classify iris flowers into two species. The model applies the sigmoid function (Eq. 2.14) to a linear combination of input features and learns parameters by minimizing the **cross-entropy cost function** (Eq. 2.24) through gradient descent (Eq. 2.27).

### Prepare IRIS Dataset

The Iris dataset contains 150 samples with four features. For binary classification, we select two species—setosa ($y=0$) and versicolor ($y=1$)—and use all four features as the input vector $\mathbf{x} \in \mathbb{R}^4$.

In [1]:
import numpy as np
import pandas as pd

from sklearn import __version__ as sklearn_version

print('NumPy version:', np.__version__)
print('Pandas version:', pd.__version__)
print('scikit-learn version:', sklearn_version)

NumPy version: 2.0.2
Pandas version: 2.2.2
scikit-learn version: 1.6.1


In [2]:
from sklearn.datasets import load_iris

iris = load_iris()

# iris.data columns: sepal length, sepal width, petal length, petal width (all in cm)
# iris.target: species label (0=setosa, 1=versicolor, 2=virginica)
iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
iris_tf = pd.DataFrame(data=iris.target, columns=['species'])

# Convert to binary labels: versicolor=1, others=0
def converter(species):
    return 1 if species == 1 else 0

iris_tf['species'] = iris_tf['species'].apply(converter)

# Remove virginica samples (indices 100-149) for binary classification
REMOVE_virginica = True

if REMOVE_virginica:
    iris_df = iris_df.drop(labels=range(100,150), axis=0)
    iris_tf = iris_tf.drop(labels=range(100,150), axis=0)

vX = iris_df.to_numpy()
vY = np.reshape(iris_tf.to_numpy(), -1)

### Presenting Dataset Samples

In [3]:
iris_df.describe()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
count,100.000000,100.000000,100.000000,100.000000
mean,5.471000,3.099000,2.861000,0.786000
std,0.641698,0.478739,1.449549,0.565153
min,4.300000,2.000000,1.000000,0.100000
25%,5.000000,2.800000,1.500000,0.200000
50%,5.400000,3.050000,2.450000,0.800000
75%,5.900000,3.400000,4.325000,1.300000
max,7.000000,4.400000,5.100000,1.800000


In [4]:
print(vY)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]


### Splitting Data for Training and Testing

In [5]:
from sklearn.model_selection import train_test_split

# Split into 80% train and 20% test
X_train, X_test, y_train, y_test = train_test_split(vX, vY, test_size=0.20, random_state=101)

### Numerically Stable Sigmoid Function

The sigmoid (logistic) function (Eq. 2.14) is defined as:

$$ \sigma(z) = \frac{1}{1 + e^{-z}} = \frac{e^{z}}{1 + e^{z}} $$

A direct implementation of $1/(1+e^{-z})$ causes overflow when $z$ is a large negative number. To avoid this, we use the first form for $z \geq 0$ and the second form for $z < 0$. Complete the code **without using an `if` statement**.

In [ ]:
def mySigmoid(x):

    ### START CODE HERE ###

    # Boolean mask: True where x >= 0
    positive = None
    # Separate positive and negative inputs to avoid overflow
    x_p = None
    x_n = None
    # Use 1/(1+exp(-z)) for z>=0, exp(z)/(1+exp(z)) for z<0
    x = None

    ### END CODE HERE ###
    
    return x

In [7]:
# Verify: sigmoid(0)=0.5, sigmoid(1000)≈1.0, sigmoid(-1000)≈0.0
mySigmoid(np.array([0.0, 1000.0, -1000.0]))

array([0.5, 1. , 0. ])

### Logistic Regression

The hypothesis function applies the sigmoid to a linear predictor (Eq. 2.19):

$$ h_{\mathbf{\theta}}(\mathbf{x}^{(i)}) = \sigma(\mathbf{w}^T \mathbf{x}^{(i)} + b) $$

The **cross-entropy** cost function (Eq. 2.24) and its equivalent log-loss form (Eq. 2.25) are:

$$ J = -\frac{1}{N} \sum_{i=1}^{N} \left( y^{(i)} \log h(\mathbf{x}^{(i)}) + (1 - y^{(i)}) \log(1 - h(\mathbf{x}^{(i)})) \right)
= \frac{1}{N} \sum_{i=1}^{N} \left( \log(1+e^{\mathbf{w}^T \mathbf{x}^{(i)} + b}) - y^{(i)}(\mathbf{w}^T \mathbf{x}^{(i)} + b) \right) $$

The gradients (Eq. 2.26) take the same form as linear regression:

$$\frac{\partial J}{\partial \mathbf{w}} = \frac{1}{N} \sum_{i=1}^{N} \left( h(\mathbf{x}^{(i)}) - y^{(i)} \right) \cdot \mathbf{x}^{(i)}, \qquad
\frac{\partial J}{\partial b} = \frac{1}{N} \sum_{i=1}^{N} \left( h(\mathbf{x}^{(i)}) - y^{(i)} \right)$$

#### Define parameter container

In [8]:
class myParameters:
    def __init__(self):
        self.weight = np.zeros((4,))  # weight vector w for 4 input features
        self.bias = 0.0               # bias scalar b

#### Train model with gradient descent

In [ ]:
# Initialize parameters (w=0, b=0)
w = myParameters()

# Set hyperparameters
alpha = 0.001     # learning rate
n_epochs = 500    # number of training iterations

for epoch in range(n_epochs):
    
    ### START CODE HERE ###

    # Compute linear predictor: z = w^T * x + b
    y_lin  = None
    # Apply sigmoid to obtain predicted probability (Eq. 2.19)
    y_hat  = None

    # Compute prediction error: (h(x) - y)
    ydiff  = None
    # Update weights and bias using gradient descent (Eq. 2.26, 2.27)
    w.weight = None
    w.bias = None

    # Compute cross-entropy loss in log-loss form (Eq. 2.25)
    loss_J = None

    ### END CODE HERE ###

    if (epoch) % 100 == 0:
        print('Epoch: %5d,  loss: %10.8f' % (epoch, loss_J))

print('Epoch: %5d,  loss: %10.8f' % (epoch, loss_J))

Epoch:     0,  loss: 0.69314718
Epoch:   100,  loss: 0.63380699
Epoch:   200,  loss: 0.59442516
Epoch:   300,  loss: 0.55961789
Epoch:   400,  loss: 0.52791238
Epoch:   499,  loss: 0.49920716


In [10]:
if alpha==0.001:
    res = [w.bias, w.weight[0], w.weight[1], w.weight[2], w.weight[3]]
    exp = [-0.01867362704808352, -0.0068206056806458006, -0.1326556030622948, 0.24802972240274482, 0.10397064127109808]
    if np.allclose(res,exp): print('test passed.')
    else: print('test failed.')
else: print('test is designed for alpha=0.001.')

test passed.


### Evaluate Model Performance

Apply the trained model to the test set. The predicted probability is converted to a class label using a threshold of 0.5.

In [ ]:
def my_predict(w, X_test):
    
    ### START CODE HERE ###

    # Compute linear predictor
    y_lin  = None
    # Apply sigmoid to obtain predicted probability
    y_pred = None

    ### END CODE HERE ###

    return y_pred

from sklearn.metrics import accuracy_score

def decision(x):
    return np.where(x>=0.5, 1, 0)

y_prob = my_predict(w, X_test)
y_pred = decision(y_prob)

accuracy_score(y_pred, y_test)

1.0

In [12]:
if alpha==0.001:
    if np.allclose(y_pred, y_test): print('test passed.')
    else: print('test failed.')
else: print('test is designed for alpha=0.001.')

test passed.


### Comparison with scikit-learn

scikit-learn's `LogisticRegression` uses an optimized solver with L2 regularization by default.

In [13]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()

# Fit the model
lr.fit(X_train, y_train)

# Predict on the test set
s_pred = lr.predict(X_test)

accuracy_score(s_pred, y_test)

1.0

### Test Model with a Random Sample

Compare predictions from our gradient descent implementation and scikit-learn on a single test sample.

In [14]:
idx = np.random.randint(X_test.shape[0])
test_in = np.expand_dims(X_test[idx], axis=0)

species = ['setosa', 'versicolor']

y_pred = decision(my_predict(w, test_in))
s_pred = lr.predict(test_in)

print('My prediction for Iris Species:', species[y_pred[0]])
print('SK prediction for Iris Species:', species[s_pred[0]])
print('Actual Iris Species:', species[y_test[idx]])

My prediction for Iris Species: setosa
SK prediction for Iris Species: setosa
Actual Iris Species: setosa


(c) 2026 S. W. Lee